In [1]:
import sys
print(sys.executable)

/Users/dominikpfeifer/Projects/german-law-obligations/.venv/bin/python


In [2]:
import requests
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

URL = "https://www.gesetze-im-internet.de/gii-toc.xml"
HEADERS = {"User-Agent": "german-law-obligations (learning project; github.com/pfeifer99)"}

response = requests.get(URL, headers=HEADERS, timeout=30)

print(response.status_code)
print(response.headers.get("Content-Type"))
print(len(response.content), "Bytes")


200
application/xml
1291178 Bytes


In [3]:
(DATA / "gii-toc.xml").write_bytes(response.content)
print(response.text[:1500])

<?xml version="1.0" encoding="UTF-8" ?><!DOCTYPE items SYSTEM "http://www.gesetze-im-internet.de/dtd/1.0/gii-toc.dtd">

<items>

<item>
<title>Gesetz über die Ausprägung einer 1-DM-Goldmünze und die Errichtung der Stiftung &quot;Geld und Währung&quot;</title>
<link>http://www.gesetze-im-internet.de/1-dm-goldm_nzg/xml.zip</link>
</item>
<item>
<title>Erstes Gesetz zur Vereinheitlichung und Neuregelung des Besoldungsrechts in Bund und Ländern</title>
<link>http://www.gesetze-im-internet.de/besvng_1/xml.zip</link>
</item>
<item>
<title>Erste Verordnung zur Durchführung des Bundes-Immissionsschutzgesetzes *)</title>
<link>http://www.gesetze-im-internet.de/bimschv_1_2010/xml.zip</link>
</item>
<item>
<title>Verordnung zur Durchführung von regelmäßigen Datenübermittlungen zwischen Meldebehörden</title>
<link>http://www.gesetze-im-internet.de/bmeldd_v_1_2015/xml.zip</link>
</item>
<item>
<title>Erste Durchführungsverordnung zur Betriebsordnung für Luftfahrtgerät (Anwendungsbestimmungen zu Anh

In [4]:
from lxml import etree
import pandas as pd

tree = etree.parse(str(DATA / "gii-toc.xml"))
root = tree.getroot()
print(root.tag)

items = root.findall("item")
print(len(items), "Einträge")

items
6144 Einträge


In [5]:
rows = []
for item in items:
    rows.append({
        "title": item.findtext("title"),
        "link": item.findtext("link"),
    })

toc = pd.DataFrame(rows)
toc.head()

,title,link
0,Gesetz über die Ausprägung einer 1-DM-Goldmünz...,http://www.gesetze-im-internet.de/1-dm-goldm_n...
1,Erstes Gesetz zur Vereinheitlichung und Neureg...,http://www.gesetze-im-internet.de/besvng_1/xml...
2,Erste Verordnung zur Durchführung des Bundes-I...,http://www.gesetze-im-internet.de/bimschv_1_20...
3,Verordnung zur Durchführung von regelmäßigen D...,http://www.gesetze-im-internet.de/bmeldd_v_1_2...
4,Erste Durchführungsverordnung zur Betriebsordn...,http://www.gesetze-im-internet.de/luftbodv_1_2...


In [6]:
toc["link"] = toc["link"].str.replace("http://", "https://", n=1)
toc["slug"] = toc["link"].str.split("/").str[-2]

print("Zeilen:", len(toc))
print("Eindeutige Kennungen:", toc["slug"].nunique())
print("Fehlende Titel:", toc["title"].isna().sum())

toc.to_csv(DATA / "toc.csv", index=False)

Zeilen: 6144
Eindeutige Kennungen: 6144
Fehlende Titel: 0
